# 02. Model Training, Calibration & Evaluation

### Objectives:
1. Stratified 80/20 train/test split preserving failure incidence.
2. Class weighting (`balanced_subsample`) to counter 3.39% imbalance.
3. Model training: Random Forest, Gradient Boosting, Logistic Regression.
4. Precision-Recall AUC, ROC-AUC, F1-Score, and Confusion Matrix analysis.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc
import joblib

df = pd.read_csv('../data/ai4i2020.csv')
df['Temp_diff'] = df['Process temperature [K]'] - df['Air temperature [K]']
df['Power_kW'] = (df['Torque [Nm]'] * df['Rotational speed [rpm]'] * 2 * np.pi) / 60000
df['Overstrain'] = df['Tool wear [min]'] * df['Torque [Nm]']

features = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Temp_diff', 'Power_kW', 'Overstrain']
X = df[features]
y = df['Machine failure']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train size: {len(X_train)}, Test size: {len(X_test)}')

In [ ]:
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced_subsample', random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
prec, rec, _ = precision_recall_curve(y_test, y_prob)
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')
print(f'PR-AUC: {auc(rec, prec):.4f}')

In [ ]:
joblib.dump(rf, '../models/random_forest_model.joblib')
print('Model saved successfully to models/random_forest_model.joblib')